#**Análisis del discurso sobre inmigración durante el Brexit utilizando comentarios de YouTube, embeddings semánticos y modelos de detección de toxicidad**

##Carlos F. Carreras De León

El objetivo de este trabajo es determinar la existencia e insinuación de discursos de odio hacia
personas migrantes analizando con métodos computacionales e inteligencia artificial los comentarios de
vídeos publicados en canales clave de YouTube, extraídos de declaraciones y posicionamientos públicos
de los principales líderes políticos del Reino Unido.

## Extracción de datos con YouTube API

Librería para abrir terminal/cmd que es un conjunto de herramientas de código (biblioteca de programación) que permite a los desarrolladores invocar la línea de comandos de su sistema operativo directamente desde su propio programa. Esto se utiliza principalmente para automatizar tareas, ejecutar otros programas o interactuar con el sistema.

In [1]:
pip install google-api-python-client pandas openpyxl youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 5.2 MB/s eta 0:00:00


Obtener playlists del canal BBC News

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import time
from youtube_transcript_api import YouTubeTranscriptApi

# =========================
# API KEY
# =========================

API_KEY = "YOUTUBE API KEY"

youtube = build("youtube", "v3", developerKey=API_KEY)

# =========================
# PASO 1: OBTENER PLAYLISTS DEL CANAL BBC NEWS
# =========================

CHANNEL_ID = "UC16niRr50-MSBwiO3YDb3RA"  # BBC News official channel

playlists = []
next_page = None

while True:

    request = youtube.playlists().list(
        part="snippet,contentDetails",
        channelId=CHANNEL_ID,
        maxResults=50,
        pageToken=next_page
    )

    response = request.execute()

    for item in response["items"]:

        playlists.append({
            "playlist_title": item["snippet"]["title"],
            "playlist_id": item["id"]
        })

    next_page = response.get("nextPageToken")

    if not next_page:
        break

playlists_df = pd.DataFrame(playlists)

print(playlists_df)

                    playlist_title                         playlist_id
0            In Case You Missed It  PLS3XGZxi7cBUZBV36CSoMRYGv5NpJ26Qp
1               Artemis II Mission  PLS3XGZxi7cBWc3_fSNq1x8IKgi-RBS9-j
2         The latest from BBC News  PLS3XGZxi7cBXc5taq0DkNtqMT8xhzl0zb
3                      Top Comment  PLS3XGZxi7cBXhtwLneoWbHm10qERaoaZZ
4          US-Israel war with Iran  PLS3XGZxi7cBXTWCW5xOhglIye__9Ee2ev
..                             ...                                 ...
79             Iraq war | BBC News  PLS3XGZxi7cBVP6j_EvAus0EVdo3uX8czj
80       War in Ukraine | BBC News  PLS3XGZxi7cBX_eteKOism2FNe6b5Uc4yi
81  India election 2024 | BBC News  PLS3XGZxi7cBUXpVeXN4r_GEZFzPF7fDhr
82                   UK | BBC News  PLS3XGZxi7cBVAYwasTLGkvDIYtnn43zHF
83           Explainers | BBC News  PLS3XGZxi7cBVouUe2EfoxH2qv897a3F6J

[84 rows x 2 columns]


Encuentra el playlist "2016 EU Referendum" del canal

In [3]:
target_playlist = playlists_df[
    playlists_df["playlist_title"].str.contains("2016 EU Referendum", case=False, na=False)
]

print(target_playlist)

                   playlist_title                         playlist_id
71  2016 EU referendum | BBC News  PLS3XGZxi7cBUgKlFuU7T3U6_v9mAPeJb7


Copia el playlist ID

In [4]:
PLAYLIST_ID = target_playlist.iloc[0]["playlist_id"]

Extraer videos del playlist

In [5]:
videos = []
next_page = None

while True:

    request = youtube.playlistItems().list(
        part="snippet",
        playlistId=PLAYLIST_ID,
        maxResults=50,
        pageToken=next_page
    )

    response = request.execute()

    for item in response["items"]:

        videos.append({
            "video_id": item["snippet"]["resourceId"]["videoId"],
            "title": item["snippet"]["title"],
            "published_at": item["snippet"]["publishedAt"]
        })

    next_page = response.get("nextPageToken")

    if not next_page:
        break

videos_df = pd.DataFrame(videos)

print("VIDEOS:", len(videos_df))

VIDEOS: 109


Estadísticas

In [6]:
stats = []

# Dividir video_ids en chunks de 50 para evitar los limites de la API
chunk_size = 50
video_id_chunks = [videos_df["video_id"][i:i + chunk_size].tolist() for i in range(0, len(videos_df), chunk_size)]

for chunk in video_id_chunks:
    video_ids_chunk = ",".join(chunk)

    stats_request = youtube.videos().list(
        part="snippet,statistics,contentDetails",
        id=video_ids_chunk
    )

    stats_response = stats_request.execute()

    for item in stats_response["items"]:

        s = item.get("statistics", {})
        sn = item.get("snippet", {})
        cd = item.get("contentDetails", {})

        stats.append({

            # =========================
            # IDENTIFICADOR
            # =========================
            "video_id": item["id"],

            # =========================
            # METADATOS BÁSICOS
            # =========================
            "title": sn.get("title"),
            "description": sn.get("description"),
            "published_at": sn.get("publishedAt"),
            "tags": sn.get("tags", []),

            # =========================
            # VISUALIZACIONES
            # =========================
            "views": s.get("viewCount"),

            # =========================
            # INTERACCIONES
            # =========================
            "likes": s.get("likeCount"),
            "comment_count": s.get("commentCount"),
        })

stats_df = pd.DataFrame(stats)

Extraer los comentarios

In [7]:
comments = []

for vid in stats_df["video_id"]:

    print("Comments:", vid)

    next_page = None

    while True:

        try:

            request = youtube.commentThreads().list(
                part="snippet",
                videoId=vid,
                maxResults=100,
                pageToken=next_page,
                textFormat="plainText"
            )

            response = request.execute()

            for item in response["items"]:

                c = item["snippet"]["topLevelComment"]["snippet"]

                comments.append({
                    "video_id": vid,
                    "comment_id": item["id"],
                    "comment_text": c.get("textDisplay"),
                    "comment_date": c.get("publishedAt"),
                    "comment_likes": c.get("likeCount")
                })

            next_page = response.get("nextPageToken")

            if not next_page:
                break

            time.sleep(0.3)

        except Exception as e:
            print("Error:", e)
            break

comments_df = pd.DataFrame(comments)

Comments: 0EkDQT23D3g
Comments: wkM2Ix2UZkU
Comments: 8rfG_c3HmS0
Comments: a8ZtOYXEhfM
Comments: p3qBlHqWgtY
Comments: 6ZQjUI_q7yg
Comments: Fec8Ya-MhXk
Comments: 4LCeUwQd-jU
Comments: i6JcUr13jsE
Comments: fNmPyXtaa1c
Comments: H86as_2Jr0g
Comments: -JkE1jYdTQA
Comments: y8EyHGxd4P4
Comments: odzSEsUFVLI
Comments: MVX-M7jvttY
Comments: vXuvsShIDjo
Comments: gp8xpXVi2IE
Comments: w3nzVKZ9Al0
Comments: LogfBroF21k
Comments: WUA5LjQqsvg
Comments: HvFUbT87-Jo
Comments: G5NwCxXE1Ws
Comments: L2AXPhK4WIk
Comments: NplV5lyhRfY
Comments: aFcSWfcPUPU
Comments: lPtRYmoKqc4
Comments: xVJkkl_cMfg
Comments: JKMNemCHMw8
Comments: LovzV_2_cjs
Comments: HO6MZcOQH0g
Comments: rk73t-OBEpc
Comments: ECDrYfNvj_o
Comments: rWed83IaL8Y
Comments: 03nLQjHSKkM
Comments: RkESFmODHF0
Comments: zyn9Lq5IX9E
Comments: yyVJlPHpGA0
Comments: Jvr6plbknso
Comments: 6Cvo2p6YDqw
Comments: bWAWbCWndnE
Comments: Dokwnx4U1aA
Comments: fFQyHH3TrR4
Comments: DxSz4levAxc
Comments: wZ4Z60funYc
Comments: RSRjA1bICuQ
Comments: 

Exportar en csv

In [8]:
stats_df.to_csv("bbc_eu_referendum_2016_videos.csv", index=False)
comments_df.to_csv("bbc_eu_referendum_2016_comments.csv", index=False)

print("DATASETS EXPORTADOS A CSV")

DATASETS EXPORTADOS A CSV


# Limpieza de datos

Cargar el data frame comments

In [9]:
comments = pd.read_csv("/content/bbc_eu_referendum_2016_comments.csv")

print("Comments DataFrame (first 5 rows):")
print(comments.head())

Comments DataFrame (first 5 rows):
      video_id                  comment_id  \
0  0EkDQT23D3g  UgwCWOiwHHi9NlP347R4AaABAg   
1  0EkDQT23D3g  UgyF6-DqqWSN22dpNVN4AaABAg   
2  0EkDQT23D3g  UgzknaiOSeSQvOrEhnx4AaABAg   
3  0EkDQT23D3g  Ugyq0yIGfqqClbDdwaF4AaABAg   
4  0EkDQT23D3g        Ugg5pCRUfI1emHgCoAEC   

                                        comment_text          comment_date  \
0  THE PM SHOULD LEAVE ON WTO TERMS AND NOT PAY T...  2019-02-06T23:32:14Z   
1      Bullshit, you can't negotiate with dictators,  2018-06-30T20:38:05Z   
2  unlike the eu the uk is a democratic country w...  2017-10-25T00:49:00Z   
3  We don't owe the EU anything. Just walk away, ...  2017-08-27T21:40:46Z   
4  Are Venetian banksters the eu?! \n\nDefinitely...  2017-06-26T16:08:12Z   

   comment_likes  
0              0  
1              0  
2              0  
3              0  
4              0  


Eliminar filas duplicadas en comments

In [10]:
num_duplicate_rows_all_cols = comments.duplicated(keep=False).sum()

if num_duplicate_rows_all_cols > 0:
    print(f"Se encontraron {num_duplicate_rows_all_cols} filas duplicadas en el DataFrame 'comments' (considerando todas las columnas).")
else:
    print("No se encontraron filas duplicadas en el DataFrame 'comments' (considerando todas las columnas).")

Se encontraron 5516 filas duplicadas en el DataFrame 'comments' (considerando todas las columnas).


In [11]:
print(f"Número de filas antes de eliminar duplicados (todas las columnas): {len(comments)}")

comments_before_dedupe_all_cols = len(comments)

comments.drop_duplicates(inplace=True)

print(f"Número de filas después de eliminar duplicados (todas las columnas): {len(comments)}")
print(f"Se eliminaron {comments_before_dedupe_all_cols - len(comments)} filas duplicadas (considerando todas las columnas).")

Número de filas antes de eliminar duplicados (todas las columnas): 23298
Número de filas después de eliminar duplicados (todas las columnas): 20540
Se eliminaron 2758 filas duplicadas (considerando todas las columnas).


Cargar el data frame videos

In [12]:
videos = pd.read_csv("/content/bbc_eu_referendum_2016_videos.csv")

print("Videos DataFrame (first 5 rows):")
print(videos.head())

Videos DataFrame (first 5 rows):
      video_id                                              title  \
0  0EkDQT23D3g  BREXIT negotiations - Tusk: UK offer for EU ci...   
1  wkM2Ix2UZkU  Nick Clegg: PM should 'delay' triggering Artic...   
2  8rfG_c3HmS0  Boris Johnson's previously unpublished 'pro-EU...   
3  a8ZtOYXEhfM  PM Theresa May to trigger Brexit (Article 50) ...   
4  p3qBlHqWgtY  Brexit: 'British people made bad decision on E...   

                                         description          published_at  \
0  UK offer for EU citizens "below expectations" ...  2017-06-23T12:55:55Z   
1  A group of MPs wants a Commons vote on any dea...  2016-10-16T09:43:55Z   
2  Boris Johnson said the UK remaining in the EU ...  2016-10-16T08:13:21Z   
3  The prime minister tells the Andrew Marr Show ...  2016-10-02T09:28:11Z   
4  Italy's PM, Matteo Renzi, says it will be "imp...  2016-09-29T10:06:51Z   

                                                tags    views  likes  \
0  ['brexit

Eliminar filas duplicadas en videos

In [13]:
# Verificar videos duplicados
duplicate_videos = videos[videos.duplicated(subset=['video_id'], keep=False)]

if not duplicate_videos.empty:
    print(f"Se encontraron {len(duplicate_videos)} videos duplicados en el DataFrame 'videos'.")
else:
    print("No se encontraron videos duplicados en el DataFrame 'videos' basado en el 'video_id'.")

Se encontraron 52 videos duplicados en el DataFrame 'videos'.


In [14]:
print(f"Número de filas antes de eliminar duplicados: {len(videos)}")

videos_before_dedupe = len(videos)

videos.drop_duplicates(subset=['video_id'], keep='first', inplace=True)

print(f"Número de filas después de eliminar duplicados: {len(videos)}")
print(f"Se eliminaron {videos_before_dedupe - len(videos)} filas duplicadas.")

Número de filas antes de eliminar duplicados: 109
Número de filas después de eliminar duplicados: 83
Se eliminaron 26 filas duplicadas.


Descrición del data frame comments

In [15]:
comments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20540 entries, 0 to 20930
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       20540 non-null  object
 1   comment_id     20540 non-null  object
 2   comment_text   20539 non-null  object
 3   comment_date   20540 non-null  object
 4   comment_likes  20540 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 962.8+ KB


Convertir variable comment_date a fecha

In [16]:
comments['comment_date'] = pd.to_datetime(comments['comment_date']).dt.strftime('%d/%m/%Y')

print("Comments DataFrame with updated 'comment_date' format (first 5 rows):")
print(comments.head())

Comments DataFrame with updated 'comment_date' format (first 5 rows):
      video_id                  comment_id  \
0  0EkDQT23D3g  UgwCWOiwHHi9NlP347R4AaABAg   
1  0EkDQT23D3g  UgyF6-DqqWSN22dpNVN4AaABAg   
2  0EkDQT23D3g  UgzknaiOSeSQvOrEhnx4AaABAg   
3  0EkDQT23D3g  Ugyq0yIGfqqClbDdwaF4AaABAg   
4  0EkDQT23D3g        Ugg5pCRUfI1emHgCoAEC   

                                        comment_text comment_date  \
0  THE PM SHOULD LEAVE ON WTO TERMS AND NOT PAY T...   06/02/2019   
1      Bullshit, you can't negotiate with dictators,   30/06/2018   
2  unlike the eu the uk is a democratic country w...   25/10/2017   
3  We don't owe the EU anything. Just walk away, ...   27/08/2017   
4  Are Venetian banksters the eu?! \n\nDefinitely...   26/06/2017   

   comment_likes  
0              0  
1              0  
2              0  
3              0  
4              0  


Descrición del data frame videos

In [17]:
videos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 83 entries, 0 to 94
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       83 non-null     object
 1   title          83 non-null     object
 2   description    83 non-null     object
 3   published_at   83 non-null     object
 4   tags           83 non-null     object
 5   views          83 non-null     int64 
 6   likes          83 non-null     int64 
 7   comment_count  83 non-null     int64 
dtypes: int64(3), object(5)
memory usage: 5.8+ KB


Convertir variable video_date a fecha

In [18]:
videos = videos.rename(columns={'published_at': 'video_date'})
videos['video_date'] = pd.to_datetime(videos['video_date']).dt.strftime('%d/%m/%Y')

print("Videos DataFrame with updated 'video_date' format (first 5 rows):")
print(videos.head())

Videos DataFrame with updated 'video_date' format (first 5 rows):
      video_id                                              title  \
0  0EkDQT23D3g  BREXIT negotiations - Tusk: UK offer for EU ci...   
1  wkM2Ix2UZkU  Nick Clegg: PM should 'delay' triggering Artic...   
2  8rfG_c3HmS0  Boris Johnson's previously unpublished 'pro-EU...   
3  a8ZtOYXEhfM  PM Theresa May to trigger Brexit (Article 50) ...   
4  p3qBlHqWgtY  Brexit: 'British people made bad decision on E...   

                                         description  video_date  \
0  UK offer for EU citizens "below expectations" ...  23/06/2017   
1  A group of MPs wants a Commons vote on any dea...  16/10/2016   
2  Boris Johnson said the UK remaining in the EU ...  16/10/2016   
3  The prime minister tells the Andrew Marr Show ...  02/10/2016   
4  Italy's PM, Matteo Renzi, says it will be "imp...  29/09/2016   

                                                tags    views  likes  \
0  ['brexit', 'brexit talks', 'brexit 

# Análisis exploratorio

In [19]:
comments['comment_date_dt'] = pd.to_datetime(comments['comment_date'], format='%d/%m/%Y')

min_comment_date = comments['comment_date_dt'].min()
max_comment_date = comments['comment_date_dt'].max()

print(f"Minimum comment date: {min_comment_date.strftime('%d/%m/%Y')}")
print(f"Maximum comment date: {max_comment_date.strftime('%d/%m/%Y')}")

Minimum comment date: 19/02/2016
Maximum comment date: 02/06/2026


Hay comentarios desde el 19/02/2016 hasta el 02/06/2026

In [80]:
import plotly.graph_objects as go
import pandas as pd

# Remuestrear comentarios por mes y contar el número de comentarios
monthly_comments = comments.set_index('comment_date_dt').resample('ME').size()

# Nombres de meses en español para etiquetas personalizadas
spanish_month_names = {
    1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'
}

# Filtrar cada tercer mes
filtered_dates = monthly_comments.index[::3]
filtered_labels = [f"{spanish_month_names[date.month]} {date.year}" for date in filtered_dates]

# Crear el gráfico de líneas de Plotly
fig = go.Figure(data=go.Scatter(
    x=monthly_comments.index,
    y=monthly_comments.values,
    mode='lines'
))

fig.update_layout(
    title_text='Número Total de Comentarios por Mes',
    xaxis_title='Fecha',
    yaxis_title='Número de Comentarios',
    xaxis={
        'tickmode': 'array',
        'tickvals': filtered_dates,
        'ticktext': filtered_labels,
        'tickangle': 45
    },
    hovermode='x unified',
    title_x=0.5, # Centrar el título
    height=800, # Aumentar la altura general del gráfico para dar más espacio
    margin=dict(t=150, b=300), # Aumentar el margen superior e inferior para el título y la nota
    annotations=[
        dict(
            text='Nota: Elaboración propia con datos extraidos del playlist "2016 EU Referendum" del canal de YouTube de la BBC News en fecha 04/06/2026.',
            xref='paper', yref='paper',
            x=0.5, y=-0.35, # Ajustar la posición vertical de la nota para que sea visible y esté más abajo
            showarrow=False,
            font=dict(size=10),
            xanchor='center', yanchor='top'
        )
    ]
)

fig.show()

El primer pico de comentarios fue en junio de 2016 cuando se hizo el referendum en Reino Unido, el segundo pico más alto fue en enero de 2021 cuando entra en efecto el Brexit y el tercer pico más alto fue en mayo de 2019 con la reununcia de la Primera Ministra Theresa May.

In [21]:
videos['video_date_dt'] = pd.to_datetime(videos['video_date'], format='%d/%m/%Y')

min_date = videos['video_date_dt'].min()
max_date = videos['video_date_dt'].max()

print(f"Minimum video date: {min_date.strftime('%d/%m/%Y')}")
print(f"Maximum video date: {max_date.strftime('%d/%m/%Y')}")

Minimum video date: 19/02/2016
Maximum video date: 23/06/2017


In [22]:
unique_video_dates = videos['video_date_dt'].dt.normalize().unique()
unique_video_dates_sorted = pd.Series(unique_video_dates).sort_values(ascending=False)

print("Unique video dates (descending order):")
for date in unique_video_dates_sorted:
    print(date.strftime('%d/%m/%Y'))

Unique video dates (descending order):
23/06/2017
16/10/2016
02/10/2016
29/09/2016
18/08/2016
02/07/2016
24/06/2016
22/06/2016
21/06/2016
20/06/2016
19/06/2016
18/06/2016
17/06/2016
16/06/2016
15/06/2016
14/06/2016
12/06/2016
10/06/2016
09/06/2016
07/06/2016
05/06/2016
02/06/2016
27/05/2016
26/05/2016
25/05/2016
22/05/2016
20/05/2016
19/05/2016
17/05/2016
16/05/2016
15/05/2016
13/05/2016
12/05/2016
09/05/2016
03/05/2016
24/04/2016
22/04/2016
19/04/2016
17/04/2016
15/04/2016
14/04/2016
10/04/2016
07/04/2016
13/03/2016
10/03/2016
24/02/2016
22/02/2016
21/02/2016
20/02/2016
19/02/2016


Los videos del playlist tienen fecha desde el 19/02/2016 hasta 16/10/2016. Después se da un salto al año siguiente e incluye un video de 23/06/2017.

###Top videos con más comentarios

In [23]:
# Asegurar que 'comment_count' sea numérico y manejar posibles valores faltantes
videos['comment_count'] = pd.to_numeric(videos['comment_count'], errors='coerce').fillna(0)

# Ordenar videos por 'comment_count' en orden descendente
most_commented_videos = videos.sort_values(by='comment_count', ascending=False)

# Obtener los 10 videos principales y crear una copia adecuada
top_10_videos = most_commented_videos.head(10).copy()

# Calcular la suma total de todos los conteos de comentarios del DataFrame 'videos' original
total_comments_sum = videos['comment_count'].sum()

# Calcular el porcentaje del total de comentarios para cada uno de los 10 videos principales
top_10_videos['percentage_of_total_comments'] = (top_10_videos['comment_count'] / total_comments_sum) * 100

# Formatear la columna de porcentaje en una nueva columna para evitar el FutureWarning
top_10_videos['percentage_of_total_comments_formatted'] = top_10_videos['percentage_of_total_comments'].round(2).astype(str) + '%'

# Preparar los datos para mostrar seleccionando las columnas relevantes y creando una nueva copia
table_data = top_10_videos[['video_id', 'title', 'comment_count', 'percentage_of_total_comments_formatted']].copy()

# Renombrar columnas para la presentación en español
table_data.rename(columns={
    'video_id': 'ID del Video',
    'title': 'Título',
    'comment_count': 'Frecuencia de Comentarios',
    'percentage_of_total_comments_formatted': 'Porcentaje del Total de Comentarios (%)'
}, inplace=True)

print("### Los 10 Videos con Mayor Número de Comentarios")
display(table_data)

### Los 10 Videos con Mayor Número de Comentarios


,ID del Video,Título,Frecuencia de Comentarios,Porcentaje del Total de Comentarios (%)
31,ECDrYfNvj_o,Eddie Izzard vs Nigel Farage on immigration - ...,10997,29.72%
4,p3qBlHqWgtY,Brexit: 'British people made bad decision on E...,4739,12.81%
72,32hgwUYpWbE,George Galloway annoyed by EU referendum quest...,2235,6.04%
40,Dokwnx4U1aA,What is the real cost of UK's EU membership? B...,1367,3.69%
6,Fec8Ya-MhXk,Brexit: The immigrants who voted Leave - BBC News,1207,3.26%
23,NplV5lyhRfY,How Geert Wilders views the European Union - B...,964,2.61%
12,y8EyHGxd4P4,UK votes to leave EU: What now? BBC News,950,2.57%
74,VTLzRHrGHds,Cameron warns leaving EU is a 'step into the d...,943,2.55%
35,zyn9Lq5IX9E,John Major: Leave campaign being 'deceitful an...,934,2.52%
9,fNmPyXtaa1c,Brexit: Why 7 out of 10 people in Hartlepool w...,914,2.47%


###Top videos con más views

In [24]:
# Asegurar que 'views' sea numérico y manejar posibles valores faltantes
videos['views'] = pd.to_numeric(videos['views'], errors='coerce').fillna(0)

# Ordenar videos por 'views' en orden descendente
most_viewed_videos = videos.sort_values(by='views', ascending=False)

# Obtener los 10 videos principales y crear una copia adecuada
top_10_viewed_videos = most_viewed_videos.head(10).copy()

# Calcular la suma total de todas las visualizaciones del DataFrame 'videos' original
total_views_sum = videos['views'].sum()

# Calcular el porcentaje del total de visualizaciones para cada uno de los 10 videos principales
top_10_viewed_videos['percentage_of_total_views'] = (top_10_viewed_videos['views'] / total_views_sum) * 100

# Formatear la columna de porcentaje en una nueva columna
top_10_viewed_videos['percentage_of_total_views_formatted'] = top_10_viewed_videos['percentage_of_total_views'].round(2).astype(str) + '%'

# Preparar los datos para mostrar seleccionando las columnas relevantes
table_data_views = top_10_viewed_videos[['video_id', 'title', 'views', 'percentage_of_total_views_formatted']].copy()

# Renombrar columnas para la presentación en español
table_data_views.rename(columns={
    'video_id': 'ID del Video',
    'title': 'Título',
    'views': 'Frecuencia de Visualizaciones',
    'percentage_of_total_views_formatted': 'Porcentaje del Total de Visualizaciones (%)'
}, inplace=True)

print("### Los 10 Videos con Mayor Número de Visualizaciones")
display(table_data_views)

### Los 10 Videos con Mayor Número de Visualizaciones


,ID del Video,Título,Frecuencia de Visualizaciones,Porcentaje del Total de Visualizaciones (%)
31,ECDrYfNvj_o,Eddie Izzard vs Nigel Farage on immigration - ...,2818174,27.03%
4,p3qBlHqWgtY,Brexit: 'British people made bad decision on E...,2125713,20.39%
72,32hgwUYpWbE,George Galloway annoyed by EU referendum quest...,889641,8.53%
74,VTLzRHrGHds,Cameron warns leaving EU is a 'step into the d...,493250,4.73%
40,Dokwnx4U1aA,What is the real cost of UK's EU membership? B...,309337,2.97%
9,fNmPyXtaa1c,Brexit: Why 7 out of 10 people in Hartlepool w...,208069,2.0%
6,Fec8Ya-MhXk,Brexit: The immigrants who voted Leave - BBC News,202452,1.94%
11,-JkE1jYdTQA,UK's time in the EU - BBC News,200525,1.92%
30,rk73t-OBEpc,Nigel Farage: Voters 'beginning to put two fin...,183249,1.76%
12,y8EyHGxd4P4,UK votes to leave EU: What now? BBC News,173807,1.67%


###Top videos con más likes

In [25]:
import pandas as pd

# Asegurar que 'likes' sea numérico y manejar posibles valores faltantes
videos['likes'] = pd.to_numeric(videos['likes'], errors='coerce').fillna(0)

# Ordenar videos por 'likes' en orden descendente
most_liked_videos = videos.sort_values(by='likes', ascending=False)

# Obtener los 10 videos principales y crear una copia adecuada
top_10_liked_videos = most_liked_videos.head(10).copy()

# Calcular la suma total de todos los 'me gusta' del DataFrame 'videos' original
total_likes_sum = videos['likes'].sum()

# Calcular el porcentaje del total de 'me gusta' para cada uno de los 10 videos principales
top_10_liked_videos['percentage_of_total_likes'] = (top_10_liked_videos['likes'] / total_likes_sum) * 100

# Formatear la columna de porcentaje en una nueva columna
top_10_liked_videos['percentage_of_total_likes_formatted'] = top_10_liked_videos['percentage_of_total_likes'].round(2).astype(str) + '%'

# Preparar los datos para mostrar seleccionando las columnas relevantes
table_data_likes = top_10_liked_videos[['video_id', 'title', 'likes', 'percentage_of_total_likes_formatted']].copy()

# Renombrar columnas para la presentación en español
table_data_likes.rename(columns={
    'video_id': 'ID del Video',
    'title': 'Título',
    'likes': 'Número de Me Gusta',
    'percentage_of_total_likes_formatted': 'Porcentaje del Total de Me Gusta (%)'
}, inplace=True)

print("### Los 10 Videos con Mayor Número de Me Gusta")
display(table_data_likes)

### Los 10 Videos con Mayor Número de Me Gusta


,ID del Video,Título,Número de Me Gusta,Porcentaje del Total de Me Gusta (%)
4,p3qBlHqWgtY,Brexit: 'British people made bad decision on E...,28983,38.24%
31,ECDrYfNvj_o,Eddie Izzard vs Nigel Farage on immigration - ...,10772,14.21%
72,32hgwUYpWbE,George Galloway annoyed by EU referendum quest...,6278,8.28%
23,NplV5lyhRfY,How Geert Wilders views the European Union - B...,2751,3.63%
40,Dokwnx4U1aA,What is the real cost of UK's EU membership? B...,2208,2.91%
74,VTLzRHrGHds,Cameron warns leaving EU is a 'step into the d...,2152,2.84%
35,zyn9Lq5IX9E,John Major: Leave campaign being 'deceitful an...,1915,2.53%
25,lPtRYmoKqc4,Jeremy Clarkson: 'Gut tells me I feel European...,1763,2.33%
30,rk73t-OBEpc,Nigel Farage: Voters 'beginning to put two fin...,1516,2.0%
6,Fec8Ya-MhXk,Brexit: The immigrants who voted Leave - BBC News,1436,1.89%


Fusión de data frames

In [26]:
comments = comments.merge(
    videos[['video_id', 'title', 'description', 'video_date', 'tags']],
    on='video_id',
    how='left'
)

comments.rename(columns={
    'title': 'video_title',
    'description': 'video_description',
    'video_date': 'video_date',
    'tags': 'video_tags'
}, inplace=True)

print("Comments DataFrame with merged video information (first 5 rows):")
print(comments.head())

Comments DataFrame with merged video information (first 5 rows):
      video_id                  comment_id  \
0  0EkDQT23D3g  UgwCWOiwHHi9NlP347R4AaABAg   
1  0EkDQT23D3g  UgyF6-DqqWSN22dpNVN4AaABAg   
2  0EkDQT23D3g  UgzknaiOSeSQvOrEhnx4AaABAg   
3  0EkDQT23D3g  Ugyq0yIGfqqClbDdwaF4AaABAg   
4  0EkDQT23D3g        Ugg5pCRUfI1emHgCoAEC   

                                        comment_text comment_date  \
0  THE PM SHOULD LEAVE ON WTO TERMS AND NOT PAY T...   06/02/2019   
1      Bullshit, you can't negotiate with dictators,   30/06/2018   
2  unlike the eu the uk is a democratic country w...   25/10/2017   
3  We don't owe the EU anything. Just walk away, ...   27/08/2017   
4  Are Venetian banksters the eu?! \n\nDefinitely...   26/06/2017   

   comment_likes comment_date_dt  \
0              0      2019-02-06   
1              0      2018-06-30   
2              0      2017-10-25   
3              0      2017-08-27   
4              0      2017-06-26   

                          

# Selección de comentarios relacionados con inmigración

Filtramos los comentarios que se refieren a inmigrantes medidante embeddings semánticos.

Primero instalamos la librería sentence-transformers. Esta librería fue desarrollada por UKP Lab (Technische Universität Darmstadt). Puedes encontrar más información y el código fuente en su repositorio oficial de GitHub, así como en la documentación de Hugging Face.

In [27]:
!pip install sentence-transformers

Importar la librería SentenceTransformer

In [28]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

Cargar modelo

In [29]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Definir frases de referencia

In [30]:
reference_texts = [

    "immigration in Britain",

    "migrants entering the UK",

    "foreigners in Britain",

    "border control and migration",

    "immigrants taking jobs",

    "mass migration into the UK",

    "eu migration",

    "open borders",

    "take back control",

    "foreign workers in Britain",

    "illegal arrivals",

    "eu citizens flooding Britain",

    "refugees entering Europe",

    "asylum seekers",

    "eastern Europeans",

    "too many migrants",

    "illegal immigration and border security concerns",

    "impact of immigration on jobs and wages in Britain",

    "pressure on public services due to migration",

    "economic effects of immigration in the UK",

    "immigration and national sovereignty",

]

Crear embeddings de referencia

In [31]:
reference_embeddings = model.encode(
    reference_texts,
    convert_to_tensor=True
)

Preprocesar los textos en comment_text

In [32]:
# Importar librería
import re

# Función de limpieza
def clean_comments(text):

    if pd.isnull(text):
        return ""

    # lowercase
    text = text.lower()

    # eliminar urls
    text = re.sub(r'http\S+', '', text)

    # eliminar saltos de línea
    text = re.sub(r'\n', ' ', text)

    # eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text)

    # eliminar html entities
    text = text.replace("&amp;", "&")

    return text.strip()

    # Aplicar
comments["clean_comment_text"] = comments[
    "comment_text"
].apply(clean_comments)

Función para medir similitud

In [33]:
# =========================
# EMBEDDINGS DE TODOS LOS COMENTARIOS
# =========================

comment_embeddings = model.encode(

    comments["clean_comment_text"].tolist(),

    convert_to_tensor=True,

    batch_size=64,

    show_progress_bar=True
)

# =========================
# CALCULAR SIMILITUD
# =========================

similarities = util.cos_sim(
    comment_embeddings,
    reference_embeddings
)

# =========================
# OBTENER SCORE MÁS ALTO
# =========================

max_scores = similarities.max(dim=1).values

# =========================
# GUARDAR RESULTADOS
# =========================

comments["immigration_score"] = max_scores.cpu().numpy()

Batches:   0%|          | 0/321 [00:00<?, ?it/s]

Filtrar sólamente comentarios sobre inmigración

In [34]:
immigration_comments = comments[
    comments["immigration_score"] > 0.40]

In [35]:
print(f"El DataFrame 'immigration_comments' tiene {len(immigration_comments)} filas.")

El DataFrame 'immigration_comments' tiene 3474 filas.


Validación a mano de 100 comentarios elegidos aleatoriamente para ver si de verdad hablan sobre inmigración.

In [36]:
sample = comments[
    comments["immigration_score"] > 0.40
].sample(100)
sample

,video_id,comment_id,comment_text,comment_date,comment_likes,comment_date_dt,video_title,video_description,video_date,video_tags,clean_comment_text,immigration_score
4580,fNmPyXtaa1c,Ugg1SAbFBKevSXgCoAEC,the begining of the EU end....,26/06/2016,0,2016-06-26,Brexit: Why 7 out of 10 people in Hartlepool w...,BBC's Fiona Trott is in Hartlepool findng out ...,24/06/2016,"['bbc', 'hartlepool', 'hartlepool referendum',...",the begining of the eu end....,0.510429
20213,w87GNWJHtFM,UgiHYSmx5nZTK3gCoAEC,"Camron you speak for your self, mate.\nYou cer...",22/02/2016,4,2016-02-22,"EU referendum: ""The choice is in your hands"" D...",Britain will vote on whether to remain in the ...,20/02/2016,"['EU referendum', 'EU', 'referendum', 'Cameron...","camron you speak for your self, mate. you cert...",0.467418
12657,ECDrYfNvj_o,UgwqLVmr-fajBuEOSPN4AaABAg,How many immigrants in your street Izzard? ...,01/06/2019,602,2019-06-01,Eddie Izzard vs Nigel Farage on immigration - ...,UKIP leader Nigel Farage and comedian Eddie Iz...,10/06/2016,"['bbc', 'eddie izzard', 'nigel farage', 'izzar...",how many immigrants in your street izzard? im ...,0.458163
20,0EkDQT23D3g,UgjvvW4T-B4hlHgCoAEC,Let's Hard Brexit quick - I've had enough of t...,23/06/2017,1,2017-06-23,BREXIT negotiations - Tusk: UK offer for EU ci...,"UK offer for EU citizens ""below expectations"" ...",23/06/2017,"['brexit', 'brexit talks', 'brexit talks bruss...",let's hard brexit quick - i've had enough of t...,0.474997
14707,RkESFmODHF0,UggUr1EnRL17Q3gCoAEC,The UK was never really engaged in the Union a...,09/06/2016,1,2016-06-09,Boris Johnson: UK population will rise 'inexor...,Former Mayor of London and Leave supporter Bor...,05/06/2016,"['BBC News', 'BBC', 'News', 'Brexit', 'Boris J...",the uk was never really engaged in the union a...,0.408643
...,...,...,...,...,...,...,...,...,...,...,...,...
10839,ECDrYfNvj_o,Ugz-qeCYk-ZerPJPgA54AaABAg,Izard to find Izard funny\nNigel been proved c...,17/01/2022,0,2022-01-17,Eddie Izzard vs Nigel Farage on immigration - ...,UKIP leader Nigel Farage and comedian Eddie Iz...,10/06/2016,"['bbc', 'eddie izzard', 'nigel farage', 'izzar...",izard to find izard funny nigel been proved co...,0.440820
3491,Fec8Ya-MhXk,UgzBkba9bh9IaoL-MBB4AaABAg,"The lady is wrong, immigrants have been coming...",09/04/2022,1,2022-04-09,Brexit: The immigrants who voted Leave - BBC News,Many would assume that those who had migrated ...,02/07/2016,"['bbc', 'bbc news', 'cameron resigns', 'david ...","the lady is wrong, immigrants have been coming...",0.605856
14268,ECDrYfNvj_o,Ugi9cmm8VVu7W3gCoAEC,it's amazing how arrogant the leave campaigner...,16/06/2016,0,2016-06-16,Eddie Izzard vs Nigel Farage on immigration - ...,UKIP leader Nigel Farage and comedian Eddie Iz...,10/06/2016,"['bbc', 'eddie izzard', 'nigel farage', 'izzar...",it's amazing how arrogant the leave campaigner...,0.480172
19994,ux_1syjelsU,UghSzI07BL2Q63gCoAEC,"TL; DL: Upon re-negotiation, we now have a spe...",21/02/2016,0,2016-02-21,David Cameron calls EU referendum for June (36...,Watch in 360 degree video as PM David Cameron ...,20/02/2016,"['360', '360 video', '360 degree', '360 degree...","tl; dl: upon re-negotiation, we now have a spe...",0.485370


# Análisis de Toxicidad

La librería detoxify fue creada por Unitary AI. Puedes encontrar más información sobre ella en su repositorio de GitHub o en la documentación oficial de la librería, usualmente en PyPI o en el sitio web de Unitary AI.

La toxicidad, tal como la calcula el modelo Detoxify que estamos utilizando, se mide en una escala entre 0 y 1. Un valor más cercano a 1 indica una mayor probabilidad de que el comentario sea tóxico, mientras que un valor más cercano a 0 indica una baja probabilidad de toxicidad.

In [37]:
!pip install detoxify

Aplicar el modelo

In [38]:
from detoxify import Detoxify

model = Detoxify("original")

immigration_comments["toxicity"] = immigration_comments[
    "clean_comment_text"
].apply(lambda x: model.predict(x)["toxicity"])

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:07<00:00, 58.6MB/s]


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/tmp/ipykernel_3838/379443568.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



¿Cuántos comentarios contienen discursos de odio?

In [39]:
immigration_comments["toxicity"].describe()

,toxicity
count,3474.000000
mean,0.151063
std,0.283606
min,0.000515
25%,0.001078
50%,0.007333
75%,0.115171
max,0.998790


Una media de 0.151063 para la toxicidad en immigration_comments significa que, en promedio, los comentarios relacionados con inmigración en tu DataFrame tienen un nivel de toxicidad bajo a moderado. Es importante recordar que esto es solo un promedio y que, hay comentarios con toxicidad muy baja (mínimo 0.000515) y otros con toxicidad muy alta (máximo 0.998790).

El 75% de los comentarios tienen una puntuación de toxicidad inferior a 0.115171, lo que sugiere que la mayoría de los comentarios no son altamente tóxicos. Sin embargo, la desviación estándar (0.284015) es relativamente alta en comparación con la media, lo que indica una dispersión considerable en los puntajes de toxicidad, es decir, hay una mezcla de comentarios no tóxicos y muy tóxicos, como ya hemos identificado con el high_toxic DataFrame.

In [81]:
import plotly.graph_objects as go
import pandas as pd

# Definir las categorías de toxicidad
def categorize_toxicity(score):
    if score < 0.4:
        return 'Baja'
    elif 0.4 <= score <= 0.7:
        return 'Media'
    else:
        return 'Alta'

# Aplicar la función para crear la nueva columna 'toxicity_level'
immigration_comments['toxicity_level'] = immigration_comments['toxicity'].apply(categorize_toxicity)

# Contar el número de comentarios por nivel de toxicidad
toxicity_counts = immigration_comments['toxicity_level'].value_counts().reset_index()
toxicity_counts.columns = ['Nivel de Toxicidad', 'Número de Comentarios']

# Calcular el porcentaje de cada categoría
total_comments = len(immigration_comments)
toxicity_counts['Porcentaje'] = (toxicity_counts['Número de Comentarios'] / total_comments) * 100

# Ordenar las categorías para una mejor visualización
toxicity_order = ['Baja', 'Media', 'Alta']
toxicity_counts['Nivel de Toxicidad'] = pd.Categorical(toxicity_counts['Nivel de Toxicidad'], categories=toxicity_order, ordered=True)
toxicity_counts = toxicity_counts.sort_values('Nivel de Toxicidad')

# Crear el gráfico de barras
fig = go.Figure(
    go.Bar(
        x=toxicity_counts['Nivel de Toxicidad'],
        y=toxicity_counts['Número de Comentarios'],
        text=[f'{p:.2f}%' for p in toxicity_counts['Porcentaje']], # Etiquetas de porcentaje
        textposition='outside',
        textfont_weight='bold' # Poner las etiquetas de porcentaje en negrita
    )
)

fig.update_layout(
    title_text='Distribución de Comentarios sobre Inmigrantes por Nivel de Toxicidad',
    xaxis_title='Nivel de Toxicidad',
    yaxis_title='Número de Comentarios',
    height=500,
    width=800,
    title_x=0.5, # Centrar el título
    plot_bgcolor='white', # Quitar el fondo
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor='black',
        tickfont=dict(weight='bold') # Poner las etiquetas del eje x en negrita
    ),
    yaxis=dict(showgrid=False, showline=True, linecolor='black'), # Ocultar cuadrícula y mostrar línea en eje y
    annotations=[
        dict(
            text='Nota: Elaboración propia con datos extraídos del playlist "2016 EU Referendum" del canal de YouTube de BBC News en fecha 04/06/2026.',
            xref='paper', yref='paper',
            x=0.5, y=-0.15, # Ajustar la posición vertical si es necesario
            showarrow=False,
            font=dict(size=10),
            xanchor='center', yanchor='top'
        )
    ]
)

fig.show()

/tmp/ipykernel_3838/1858056103.py:14: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [40]:
immigration_comments.to_csv("immigration_comments.csv", index=False)
print("DataFrame 'immigration_comments' guardado como immigration_comments.csv")

DataFrame 'immigration_comments' guardado como immigration_comments.csv


¿Cuántos comentarios son altamente tóxicos?

In [41]:
high_toxic = immigration_comments[
    immigration_comments["toxicity"] > 0.70
]
print(f"Hay {len(high_toxic)} comentarios altamente tóxicos.")

Hay 351 comentarios altamente tóxicos.


In [42]:
high_toxic.to_csv("high_toxic.csv", index=False)
print("DataFrame 'high_toxic' guardado como high_toxic.csv")

DataFrame 'high_toxic' guardado como high_toxic.csv


Porcentaje de comentarios altamente tóxicos sobre el total de comentarios sobre inmigrantes

In [43]:
len(high_toxic) / len(immigration_comments)

0.10103626943005181

Entre los 3474 comentarios relacionados con inmigrantes el 10% son altamente tóxicos.

In [44]:
high_toxic

,video_id,comment_id,comment_text,comment_date,comment_likes,comment_date_dt,video_title,video_description,video_date,video_tags,clean_comment_text,immigration_score,toxicity
13,0EkDQT23D3g,UgjTEth6AId6RngCoAEC,Fuck the EU and their yanky USA terrorism,23/06/2017,1,2017-06-23,BREXIT negotiations - Tusk: UK offer for EU ci...,"UK offer for EU citizens ""below expectations"" ...",23/06/2017,"['brexit', 'brexit talks', 'brexit talks bruss...",fuck the eu and their yanky usa terrorism,0.439182,0.998148
20,0EkDQT23D3g,UgjvvW4T-B4hlHgCoAEC,Let's Hard Brexit quick - I've had enough of t...,23/06/2017,1,2017-06-23,BREXIT negotiations - Tusk: UK offer for EU ci...,"UK offer for EU citizens ""below expectations"" ...",23/06/2017,"['brexit', 'brexit talks', 'brexit talks bruss...",let's hard brexit quick - i've had enough of t...,0.474997,0.897319
22,0EkDQT23D3g,UggNhhN-ba7BLngCoAEC,"""Below expectations"" ok then great you take yo...",23/06/2017,1,2017-06-23,BREXIT negotiations - Tusk: UK offer for EU ci...,"UK offer for EU citizens ""below expectations"" ...",23/06/2017,"['brexit', 'brexit talks', 'brexit talks bruss...","""below expectations"" ok then great you take yo...",0.417165,0.969649
32,0EkDQT23D3g,Ughc_zSnvfX5GXgCoAEC,FOOK the EU !!!!!,23/06/2017,0,2017-06-23,BREXIT negotiations - Tusk: UK offer for EU ci...,"UK offer for EU citizens ""below expectations"" ...",23/06/2017,"['brexit', 'brexit talks', 'brexit talks bruss...",fook the eu !!!!!,0.445349,0.743327
34,0EkDQT23D3g,Ugj2s-WEUtQ23HgCoAEC,so what does this idiot want? .. he wants all ...,23/06/2017,1,2017-06-23,BREXIT negotiations - Tusk: UK offer for EU ci...,"UK offer for EU citizens ""below expectations"" ...",23/06/2017,"['brexit', 'brexit talks', 'brexit talks bruss...",so what does this idiot want? .. he wants all ...,0.568366,0.790707
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20275,nKJjLH9Mp9c,UggoC_31SbvbWngCoAEC,Eu leaders worried that if Britain leaves Euro...,20/02/2016,0,2016-02-20,EU: UK to have 'the best of both worlds' says ...,There is unanimous support for a deal between ...,19/02/2016,"['Donald Tusk', 'David Cameron', 'Cameron', 'E...",eu leaders worried that if britain leaves euro...,0.455328,0.704139
20283,nKJjLH9Mp9c,UgjsMe9SbhxLtHgCoAEC,This is too easy! Even a Brit is not an idiot!...,20/02/2016,0,2016-02-20,EU: UK to have 'the best of both worlds' says ...,There is unanimous support for a deal between ...,19/02/2016,"['Donald Tusk', 'David Cameron', 'Cameron', 'E...",this is too easy! even a brit is not an idiot!...,0.408753,0.964896
20289,nKJjLH9Mp9c,Ughg-AK47QzO8HgCoAEC,"SO WHAT!!! \nBring out the referendum, im stil...",20/02/2016,5,2016-02-20,EU: UK to have 'the best of both worlds' says ...,There is unanimous support for a deal between ...,19/02/2016,"['Donald Tusk', 'David Cameron', 'Cameron', 'E...","so what!!! bring out the referendum, im still ...",0.497020,0.932983
20431,fRvVXuZ_wjk,UggX3jb8zBLHk3gCoAEC,Obomba keep your nose out of England and fix y...,25/04/2016,0,2016-04-25,"Barack Obama on Brexit, Syria and Michelle - B...","Speaking exclusively to the BBC, US President ...",24/04/2016,"['bbc', 'Barack Obama', 'barack obama intervie...",obomba keep your nose out of england and fix y...,0.429643,0.841933


In [74]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [78]:
import json

notebook_path = "/content/drive/MyDrive/Brexit_Immigration_Discourse_and_Toxicity_Analysis.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Eliminar metadata de widgets que rompe GitHub y nbconvert
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

clean_path = "/content/drive/MyDrive/Brexit_Immigration_Discourse_and_Toxicity_Analysis_clean.ipynb"

with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Notebook limpio guardado en:", clean_path)

Notebook limpio guardado en: /content/drive/MyDrive/Brexit_Immigration_Discourse_and_Toxicity_Analysis_clean.ipynb


In [79]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Brexit_Immigration_Discourse_and_Toxicity_Analysis_clean.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Brexit_Immigration_Discourse_and_Toxicity_Analysis_clean.ipynb to html
[NbConvertApp] Writing 530334 bytes to /content/drive/MyDrive/Brexit_Immigration_Discourse_and_Toxicity_Analysis_clean.html
